# Inner Speech Dataset — пример загрузки данных

Датасет содержит записи мысленного произнесения слов на двух языках: **Russian** (12 испытуемых, 14 слов) и **Spanish** (10 испытуемых, 12 слов).  
Для каждого испытуемого доступны:
- `*.edf` / `*_edited.edf` — сырая запись ЭЭГ
- `*-epo.fif` — эпохи
- `*-clean-epo.fif` — эпохи после удаления артефактов

Каждое слово записывалось дважды (два блока): **overt** (вслух, суффикс `1`) и **inner** (мысленно, суффикс `2`).

In [1]:
import sys
sys.path.insert(0, '..')

import mne
import pandas as pd
from pathlib import Path

import innersp
from innersp import (
    load_raw, load_all_raw,
    load_epochs, load_all_epochs, load_all_languages,
    LABEL_INFO, INFORMATIVE, SERVICE, ATYPICAL,
    DEFAULT_BASE,
)

mne.set_log_level('WARNING')

C:\Users\a4431\AppData\Roaming\Python\Python39\site-packages\matplotlib\projections\__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


## 1. Справочник меток

In [2]:
display(LABEL_INFO)

print(f'\nИнформативные метки ({len(INFORMATIVE)}): {sorted(INFORMATIVE)}')
print(f'Служебные метки: {sorted(SERVICE)}')

,id_russian,id_spanish,english,russian,spanish,speech_type
label,,,,,,
BACK1,1,1.0,back,назад,atrás,overt
BACK2,2,2.0,back,назад,atrás,inner
DOWN1,3,3.0,down,вниз,abajo,overt
DOWN2,4,4.0,down,вниз,abajo,inner
FORWARD1,5,5.0,forward,вперёд,adelante,overt
FORWARD2,6,6.0,forward,вперёд,adelante,inner
GO,7,7.0,go,старт,inicio,service: block start
GZ,8,8.0,rest,покой,descanso,service: rest/baseline
LEFT1,9,9.0,left,влево,izquierda,overt



Информативные метки (14): ['BACK1', 'BACK2', 'DOWN1', 'DOWN2', 'FORWARD1', 'FORWARD2', 'LEFT1', 'LEFT2', 'NEXT1', 'NEXT2', 'RIGHT1', 'RIGHT2', 'UP1', 'UP2']
Служебные метки: ['GO', 'GZ']


## 2. Загрузка сырых EDF

In [3]:
# Один испытуемый
raw = load_raw('sub2', 'Russian')
print(raw)
print(f'Каналов: {len(raw.ch_names)}, частота: {raw.info["sfreq"]} Гц, длина: {raw.times[-1]:.1f} с')

<RawEDF | sub2_edited.edf, 37 x 1970500 (3941.0 s), ~40 kB, data not loaded>
Каналов: 37, частота: 500.0 Гц, длина: 3941.0 с


In [4]:
# Аннотации
ann_df = pd.DataFrame({
    'onset_s':     raw.annotations.onset,
    'duration_s':  raw.annotations.duration,
    'description': raw.annotations.description,
})

print('Уникальные метки:', sorted(ann_df['description'].unique()))
print(f'\nПервые 10 аннотаций:')
display(ann_df.head(10))

Уникальные метки: ['BACK1', 'BACK2', 'DOWN1', 'DOWN2', 'FORWARD1', 'FORWARD2', 'GO', 'GZ', 'LEFT1', 'LEFT2', 'NEXT1', 'NEXT2', 'RIGHT1', 'RIGHT2', 'UP1', 'UP2']

Первые 10 аннотаций:


,onset_s,duration_s,description
0,15.796,0.0,GO
1,16.796,0.0,GO
2,17.796,0.0,GO
3,19.796,0.0,GO
4,20.796,0.0,GO
5,21.796,0.0,GO
6,22.796,0.0,GO
7,23.796,0.0,GO
8,24.796,0.0,GO
9,25.796,0.0,GO


In [5]:
# Все испытуемые одного языка
all_raw_ru = load_all_raw('Russian')
print('Загружено:', list(all_raw_ru.keys()))

c:\Users\a4431\VSProjects\innersp\innersp\loader.py:59: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  result[subject] = mne.io.read_raw_edf(str(edf), preload=preload, verbose=False)


Загружено: ['sub1', 'sub10', 'sub11', 'sub12', 'sub2', 'sub3', 'sub4', 'sub5', 'sub6', 'sub7', 'sub8', 'sub9']


## 3. Загрузка эпох (FIF)

In [6]:
# Один испытуемый — чистые эпохи
epochs = load_epochs('sub2', 'Russian', clean=True)
print(epochs)
print(f'\nЭпох: {len(epochs)}')
print(f'Каналов: {len(epochs.ch_names)}')
print(f'Временное окно: [{epochs.tmin:.3f}, {epochs.tmax:.3f}] с')
print(f'Метки: {epochs.event_id}')

<EpochsFIF | 1395 events (all good), 0 – 0.998 s (baseline off), ~58 kB, data not loaded,
 'BACK1': 84
 'BACK2': 87
 'DOWN1': 83
 'DOWN2': 84
 'FORWARD1': 83
 'FORWARD2': 82
 'GO': 98
 'GZ': 89
 'LEFT1': 86
 'LEFT2': 84
 and 6 more events ...>

Эпох: 1395
Каналов: 37
Временное окно: [0.000, 0.998] с
Метки: {'BACK1': 1, 'BACK2': 2, 'DOWN1': 3, 'DOWN2': 4, 'FORWARD1': 5, 'FORWARD2': 6, 'GO': 7, 'GZ': 8, 'LEFT1': 9, 'LEFT2': 10, 'NEXT1': 11, 'NEXT2': 12, 'RIGHT1': 13, 'RIGHT2': 14, 'UP1': 15, 'UP2': 16}


In [7]:
# Выбор по условию: только inner speech
inner_labels = [l for l in epochs.event_id if l.endswith('2') and l not in SERVICE]
epochs_inner = epochs[inner_labels]
print(f'Inner speech эпох: {len(epochs_inner)}')

# Только overt speech
overt_labels = [l for l in epochs.event_id if l.endswith('1') and l not in SERVICE]
epochs_overt = epochs[overt_labels]
print(f'Overt speech эпох: {len(epochs_overt)}')

Inner speech эпох: 595
Overt speech эпох: 613


In [8]:
# Все испытуемые обоих языков
all_epochs = load_all_languages(clean=True)

rows = []
for language, subjects in all_epochs.items():
    for subject, ep in subjects.items():
        is_atypical = (language, subject) in ATYPICAL
        rows.append({
            'language': language,
            'subject':  subject,
            'n_epochs': len(ep),
            'n_channels': len(ep.ch_names),
            'sfreq_Hz': ep.info['sfreq'],
            'tmin_s':   round(ep.tmin, 3),
            'tmax_s':   round(ep.tmax, 3),
            'atypical': '⚠' if is_atypical else '',
        })

display(pd.DataFrame(rows).set_index(['language', 'subject']))

n_epochs  n_channels  sfreq_Hz  tmin_s  tmax_s atypical
language subject                                                         
Russian  sub1          864          37     500.0     0.0   0.998        ⚠
         sub10         912          37     500.0     0.0   0.998        ⚠
         sub11        1365          40     500.0     0.0   0.998         
         sub12         477          37     500.0     0.0   0.998         
         sub2         1395          37     500.0     0.0   0.998         
         sub3          912          37     500.0     0.0   0.998        ⚠
         sub4         1372          37     500.0     0.0   0.998         
         sub5          804          37     500.0     0.0   0.998        ⚠
         sub6          398          37     500.0     0.0   0.998        ⚠
         sub7         1343          37     500.0     0.0   0.998         
         sub8         1416          37     500.0     0.0   0.998         
         sub9         1343          35     250.0     0.0   0.996         
Spanish  sub0         2029          38     500.0     0.0   0.998         
         sub1         1631          38     500.0     0.0   0.998         
         sub2         2208          38     500.0     0.0   0.998         
         sub3         1101          38     500.0     0.0   0.998         
         sub4         1168          38     500.0     0.0   0.998         
         sub5         1657          38     500.0     0.0   0.998         
         sub6         1552          40     500.0     0.0   0.998         
         sub7         1620          40     500.0     0.0   0.998         
         sub8         1425          40     500.0     0.0   0.998         
         sub9         1267          40     500.0     0.0   0.998

## 4. Структура меток в файлах

Проверка соответствия реальных меток в EDF справочнику.

In [9]:
rows = []
for language in ('Russian', 'Spanish'):
    for edf in sorted((DEFAULT_BASE / language).rglob('*_edited.edf')):
        subject = edf.parent.name
        raw = mne.io.read_raw_edf(str(edf), preload=False, verbose=False)
        labels = set(raw.annotations.description)
        garbage = labels - INFORMATIVE - SERVICE
        rows.append({
            'language':       language,
            'subject':        subject,
            'all_labels':     sorted(labels),
            'garbage_labels': sorted(garbage) if garbage else None,
        })

df_check = pd.DataFrame(rows).set_index(['language', 'subject'])
display(df_check)

bad = df_check[df_check['garbage_labels'].notna()]
if bad.empty:
    print('Все файлы содержат только ожидаемые метки.')
else:
    print(f'Файлов с посторонними метками: {len(bad)}')
    display(bad)

C:\Users\a4431\AppData\Local\Temp\ipykernel_11680\1227321144.py:5: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(str(edf), preload=False, verbose=False)
C:\Users\a4431\AppData\Local\Temp\ipykernel_11680\1227321144.py:5: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(str(edf), preload=False, verbose=False)
C:\Users\a4431\AppData\Local\Temp\ipykernel_11680\1227321144.py:5: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(str(edf), preload=False, verbose=False)
C:\Users\a4431\AppData\Local\Temp\ipykernel_11680\1227321144.py:5: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(str(edf), preload=False, verbose=False)
C:\Users\a4431\AppData\Local\Temp\ipykernel_11680\1227321144.py:5: R

all_labels  \
language subject                                                      
Russian  sub1     [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub10    [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub11    [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub12    [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub2     [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub3     [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub4     [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub5     [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub6     [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub7     [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub8     [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub9     [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
Spanish  sub0     [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub1     [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub2     [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub3     [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub4     [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub5     [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub6     [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub7     [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub8     [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   
         sub9     [BACK1, BACK2, DOWN1, DOWN2, FORWARD1, FORWARD...   

                 garbage_labels  
language subject                 
Russian  sub1              None  
         sub10             None  
         sub11             None  
         sub12             None  
         sub2              None  
         sub3              None  
         sub4              None  
         sub5              None  
         sub6              None  
         sub7              None  
         sub8              None  
         sub9              None  
Spanish  sub0              None  
         sub1              None  
         sub2              None  
         sub3              None  
         sub4              None  
         sub5              None  
         sub6              None  
         sub7              None  
         sub8              None  
         sub9              None

Все файлы содержат только ожидаемые метки.
